# RBRICSwithMinority

In [1]:
# folder_name = "../RBRICS_MINORITY_MP2_DIM16_CORRECTED"
folder_name = "../STABLE/RBRICS_MINORITY_MP2_DIM32"
root_dirs_dict = {
    "Mutagenicity": f"{folder_name}/MOSE_BC",
    "hERG": f"{folder_name}/MOSE_BC",
    "BBBP": f"{folder_name}/MOSE_BC",
    "esol": f"{folder_name}/MOSE_Reg",
    "Lipophilicity": f"{folder_name}/MOSE_Reg",
    "Benzene": f"{folder_name}/MOSE_BC",
    "Alkane_Carbonyl": f"{folder_name}/MOSE_BC",
    "Fluoride_Carbonyl": f"{folder_name}/MOSE_BC",
}

# expl_lr_values = ["0001", "001", "01"]
# gnn_lr_values = ["0001", "001", "01"]
expl_lr_values = ["01"]
gnn_lr_values = ["0001", "001"]
architectures = {"GAT", "GCN", "GIN", "SAGE"}
types = {"RBRICS"}
valid_folds = [str(f) for f in range(5)]

motif_frequency_threshold = {
    "esol": 10, "BBBP": 18, "Lipophilicity": 38, "Mutagenicity": 70,
    "Benzene": 70, "Alkane_Carbonyl": 50, "Fluoride_Carbonyl": 50,
    "hERG": 90, "tox21": 70,
}


In [2]:
import os
import re
import pandas as pd
import numpy as np
from collections import defaultdict

def Sigmoid(x):
    return 1 / (1 + np.exp(-x))

# Updated to include expl_lr and gnn_lr
plotting_data = defaultdict(lambda: defaultdict(lambda: defaultdict(lambda: defaultdict(lambda: defaultdict(list)))))
training_losses = defaultdict(lambda: defaultdict(lambda: defaultdict(lambda: defaultdict(list))))
validation_losses = defaultdict(lambda: defaultdict(lambda: defaultdict(lambda: defaultdict(list))))
found_keys = set()
missing_data_records = []

for EXPL_LR in expl_lr_values:
    for GNN_LR in gnn_lr_values:
        print(f"\n=== Checking EXPLLR=0.{EXPL_LR}, GNNLR=0.{GNN_LR} ===")
        folder_pattern = re.compile(
            fr"EXPT-\d+[A-Z]*-([\w_]+)-SEED-\d+-FOLD-(\d+)-(\w+)-EXPLLR0\.{EXPL_LR}-GNNLR0\.{GNN_LR}-([\w\d]+|None)-(RBRICS|None)$"
        )

        for dataset_name, root_dir in root_dirs_dict.items():
            for fold in valid_folds:
                for arch in architectures:
                    for model_type in types:
                        folder_found = False
                        missing_files = []
                        dfs_all = []

                        if not os.path.exists(root_dir):
                            missing_files.append(f"Root directory missing: {root_dir}")
                        else:
                            for folder in os.listdir(root_dir):
                                match = folder_pattern.match(folder)
                                if not match:
                                    continue

                                dset, f, a, _, t = match.groups()
                                if (dset == dataset_name and f == fold
                                        and a == arch and t == model_type):
                                    folder_found = True
                                    folder_path = os.path.join(root_dir, folder)
                                    training_path = os.path.join(folder_path, "explainer", f"{dataset_name}.csv")
                                    result_files = [
                                        os.path.join(folder_path, f"{dataset_name}_explanation_result_with_train.csv"),
                                        os.path.join(folder_path, f"{dataset_name}_explanation_result_with_validation.csv"),
                                        os.path.join(folder_path, f"{dataset_name}_explanation_result_with_test.csv"),
                                    ]

                                    # Check for missing files
                                    if not os.path.exists(training_path):
                                        missing_files.append(training_path)
                                    for file in result_files:
                                        if not os.path.exists(file):
                                            missing_files.append(file)

                                    if not missing_files:
                                        # Read training/validation losses
                                        try:
                                            df_train = pd.read_csv(training_path)
                                            if "Train Loss" in df_train.columns and "Val Loss" in df_train.columns:
                                                training_losses[dataset_name][arch][EXPL_LR][GNN_LR].append(df_train["Train Loss"].values)
                                                validation_losses[dataset_name][arch][EXPL_LR][GNN_LR].append(df_train["Val Loss"].values)
                                        except Exception as e:
                                            print(f"Error reading {training_path}: {e}")
                                            missing_files.append(f"Error reading training CSV: {e}")

                                        # Read and process result CSVs
                                        for file in result_files:
                                            try:
                                                df = pd.read_csv(file)
                                                motif_counts = df["motif"].value_counts()
                                                df = df[df["motif"].map(motif_counts) > motif_frequency_threshold[dataset_name]]
                                                df["logit_diff"] = np.abs(Sigmoid(df["original_logit"]) - Sigmoid(df["new_logit"]))
                                                df["diff_sign"] = np.sign(Sigmoid(df["original_logit"]) - Sigmoid(df["new_logit"]))
                                                df["sigmoid_bin"] = pd.cut(df["sigmoid_importance"], bins=np.linspace(0, 1, 11), include_lowest=True)
                                                dfs_all.append(df)
                                            except Exception as e:
                                                print(f"Error reading {file}: {e}")
                                                missing_files.append(f"Error reading {file}: {e}")
                                    break  # Stop after finding matching folder

                        if not folder_found:
                            missing_files.append("Folder matching pattern not found")

                        if missing_files:
                            missing_data_records.append({
                                "Dataset": dataset_name,
                                "Architecture": arch,
                                "Type": model_type,
                                "Fold": fold,
                                "EXPL_LR": f"0.{EXPL_LR}",
                                "GNN_LR": f"0.{GNN_LR}",
                                "Missing Files": ", ".join(missing_files)
                            })
                            print(f"Missing: Dataset={dataset_name}, Arch={arch}, Type={model_type}, Fold={fold}, "
                                  f"EXPL_LR=0.{EXPL_LR}, GNN_LR=0.{GNN_LR}")
                        else:
                            # If all files exist, add to found_keys and plotting_data
                            found_keys.add((dataset_name, arch, model_type, fold, EXPL_LR, GNN_LR))
                            if dfs_all:
                                df_all = pd.concat(dfs_all, ignore_index=True)
                                plotting_data[arch][dataset_name][model_type][EXPL_LR][GNN_LR].append(df_all)
                                print(f"Collected: [{dataset_name}] {arch}-{model_type} Fold={fold} EXPL_LR=0.{EXPL_LR} GNN_LR=0.{GNN_LR} Samples={len(df_all)}")

# Merge folds for each configuration in plotting_data
for arch in plotting_data:
    for dataset in plotting_data[arch]:
        for model_type in plotting_data[arch][dataset]:
            for EXPL_LR in plotting_data[arch][dataset][model_type]:
                for GNN_LR in plotting_data[arch][dataset][model_type][EXPL_LR]:
                    plotting_data[arch][dataset][model_type][EXPL_LR][GNN_LR] = pd.concat(
                        plotting_data[arch][dataset][model_type][EXPL_LR][GNN_LR], ignore_index=True
                    )

# Save missing file report to CSV
report_df = pd.DataFrame(missing_data_records)
report_csv_path = "missing_files_report.csv"
report_df.to_csv(report_csv_path, index=False)
print(f"\n==== Missing Files Report saved to {report_csv_path} ====")



=== Checking EXPLLR=0.01, GNNLR=0.0001 ===
Collected: [Mutagenicity] GAT-RBRICS Fold=0 EXPL_LR=0.01 GNN_LR=0.0001 Samples=16504
Collected: [Mutagenicity] SAGE-RBRICS Fold=0 EXPL_LR=0.01 GNN_LR=0.0001 Samples=16504
Collected: [Mutagenicity] GCN-RBRICS Fold=0 EXPL_LR=0.01 GNN_LR=0.0001 Samples=16504
Collected: [Mutagenicity] GIN-RBRICS Fold=0 EXPL_LR=0.01 GNN_LR=0.0001 Samples=16504
Collected: [Mutagenicity] GAT-RBRICS Fold=1 EXPL_LR=0.01 GNN_LR=0.0001 Samples=16810
Collected: [Mutagenicity] SAGE-RBRICS Fold=1 EXPL_LR=0.01 GNN_LR=0.0001 Samples=16810
Collected: [Mutagenicity] GCN-RBRICS Fold=1 EXPL_LR=0.01 GNN_LR=0.0001 Samples=16810
Collected: [Mutagenicity] GIN-RBRICS Fold=1 EXPL_LR=0.01 GNN_LR=0.0001 Samples=16810
Collected: [Mutagenicity] GAT-RBRICS Fold=2 EXPL_LR=0.01 GNN_LR=0.0001 Samples=16382
Collected: [Mutagenicity] SAGE-RBRICS Fold=2 EXPL_LR=0.01 GNN_LR=0.0001 Samples=16382
Collected: [Mutagenicity] GCN-RBRICS Fold=2 EXPL_LR=0.01 GNN_LR=0.0001 Samples=16382
Collected: [Mutage

In [3]:
#Training Losses plot
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import LogLocator

arch_colors = {
    "GAT": "tab:blue",
    "GCN": "tab:orange",
    "GIN": "tab:green",
    "SAGE": "tab:red"
}

# Set global matplotlib styles for professional plots
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
    "axes.titlesize": 14,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "axes.linewidth": 1.2,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "lines.linewidth": 2.0,
})

# Ensure output directory exists
output_dir = f"{folder_name}/plots/training_val_loss_plots"
os.makedirs(output_dir, exist_ok=True)

for dataset_name in training_losses:
    for loss_dict, loss_type in zip(
        [training_losses[dataset_name], validation_losses[dataset_name]],
        ["Train Loss", "Validation Loss"]
    ):
        fig, axes = plt.subplots(len(expl_lr_values), len(gnn_lr_values),
                                 figsize=(5 * len(gnn_lr_values), 4 * len(expl_lr_values)),
                                 sharex=True, sharey=False)
        axes = np.atleast_2d(axes)  # Ensure axes is 2D

        # Precompute row min/max values for y-limits
        row_minmax_values = []
        for i, expl_lr in enumerate(expl_lr_values):
            row_min = float('inf')
            row_max = 0
            for arch in architectures:
                for gnn_lr in gnn_lr_values:
                    if (arch in loss_dict and
                        expl_lr in loss_dict[arch] and
                        gnn_lr in loss_dict[arch][expl_lr]):
                        runs = loss_dict[arch][expl_lr][gnn_lr]
                        min_len = min(len(r) for r in runs)
                        losses = np.array([r[:min_len] for r in runs])
                        mean_loss = losses.mean(axis=0)
                        std_loss = losses.std(axis=0)
                        row_min = min(row_min, np.min(mean_loss - std_loss))
                        row_max = max(row_max, np.max(mean_loss + std_loss))
            row_minmax_values.append((max(0, row_min), row_max))  # Clamp lower y-limit at 0

        for i, expl_lr in enumerate(expl_lr_values):
            for j, gnn_lr in enumerate(gnn_lr_values):
                ax = axes[i, j]
                active_architectures = []  # Track which architectures were actually plotted

                for arch in architectures:
                    # Skip GIN only for Validation Loss and GNN LR = 0.01
                    if ("validation" in loss_type.lower() and gnn_lr == "01" and arch == "GIN"):
                        continue

                    if (arch in loss_dict and
                        expl_lr in loss_dict[arch] and
                        gnn_lr in loss_dict[arch][expl_lr]):
                        runs = loss_dict[arch][expl_lr][gnn_lr]
                        min_len = min(len(r) for r in runs)
                        losses = np.array([r[:min_len] for r in runs])
                        mean_loss = losses.mean(axis=0)
                        std_loss = losses.std(axis=0)

                        epochs = np.arange(min_len)
                        ax.plot(epochs, mean_loss, label=arch, color=arch_colors[arch])
                        ax.fill_between(epochs, mean_loss - std_loss, mean_loss + std_loss,
                                        color=arch_colors[arch], alpha=0.15)
                        active_architectures.append(arch)

                # Set individual y-limits per subplot
                all_values = []
                for arch in active_architectures:
                    if (arch in loss_dict and
                        expl_lr in loss_dict[arch] and
                        gnn_lr in loss_dict[arch][expl_lr]):
                        runs = loss_dict[arch][expl_lr][gnn_lr]
                        min_len = min(len(r) for r in runs)
                        losses = np.array([r[:min_len] for r in runs])
                        mean_loss = losses.mean(axis=0)
                        std_loss = losses.std(axis=0)
                        all_values.append(mean_loss - std_loss)
                        all_values.append(mean_loss + std_loss)

                # Flatten and filter all_values
                flat_values = np.concatenate([v.ravel() for v in all_values if isinstance(v, np.ndarray) and v.size > 0], axis=0)

                ymin = max(0, np.min(flat_values))
                ymax = np.max(flat_values)
                ax.set_ylim(ymin, ymax)
               
                ax.set_yscale('log')
                ax.yaxis.set_major_locator(LogLocator(base=10.0))
                ax.yaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(1.1, 10) * 0.1, numticks=10))

                if i == len(expl_lr_values) - 1:
                    ax.set_xlabel("Epoch", fontsize=13)

                ax.tick_params(width=1.2)
                ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)

                if i == 0:
                    ax.set_title(f"GNN LR = 0.{gnn_lr}", fontsize=14, pad=10)

                if j == 0:
                    ax.annotate(f"Expl LR = 0.{expl_lr}",
                                xy=(-0.35, 0.5),
                                xycoords='axes fraction',
                                ha='center', va='center',
                                rotation=90, fontsize=14)

        
        
        # Add grid-level labels
        fig.text(0.5, 0.04, "Epoch", ha="center", va="center", fontsize=15)
        fig.text(0.04, 0.5, f"{loss_type} Log Scaled", ha="center", va="center", rotation="vertical", fontsize=15)
        fig.text(0.5, 0.97, f"{dataset_name} - {loss_type}", ha="center", va="center", fontsize=16, weight='bold')

        # Build handles only from architectures actually plotted across all subplots
        unique_active_architectures = set()
        for ax_row in axes:
            for ax in ax_row:
                unique_active_architectures.update([line.get_label() for line in ax.get_lines()])

        handles = [Line2D([0], [0], color=arch_colors[arch], label=arch)
                   for arch in unique_active_architectures if arch in arch_colors]

        fig.legend(handles=handles, title="Architecture", loc="center right",
                   bbox_to_anchor=(1.02, 0.5), fontsize=12, title_fontsize=13)
        

        fig.tight_layout(rect=[0.06, 0.06, 0.94, 0.94], pad=2.0)

        # Save figure
        out_file = f"{output_dir}/{dataset_name}_{loss_type.replace(' ', '_').lower()}_heatmap.png"
        fig.savefig(out_file, dpi=300, bbox_inches="tight")
        plt.close(fig)
        print(f"✅ Saved plot: {out_file}")


✅ Saved plot: ../STABLE/RBRICS_MINORITY_MP2_DIM16/plots/training_val_loss_plots/Mutagenicity_train_loss_heatmap.png
✅ Saved plot: ../STABLE/RBRICS_MINORITY_MP2_DIM16/plots/training_val_loss_plots/Mutagenicity_validation_loss_heatmap.png
✅ Saved plot: ../STABLE/RBRICS_MINORITY_MP2_DIM16/plots/training_val_loss_plots/hERG_train_loss_heatmap.png
✅ Saved plot: ../STABLE/RBRICS_MINORITY_MP2_DIM16/plots/training_val_loss_plots/hERG_validation_loss_heatmap.png
✅ Saved plot: ../STABLE/RBRICS_MINORITY_MP2_DIM16/plots/training_val_loss_plots/BBBP_train_loss_heatmap.png
✅ Saved plot: ../STABLE/RBRICS_MINORITY_MP2_DIM16/plots/training_val_loss_plots/BBBP_validation_loss_heatmap.png
✅ Saved plot: ../STABLE/RBRICS_MINORITY_MP2_DIM16/plots/training_val_loss_plots/esol_train_loss_heatmap.png
✅ Saved plot: ../STABLE/RBRICS_MINORITY_MP2_DIM16/plots/training_val_loss_plots/esol_validation_loss_heatmap.png
✅ Saved plot: ../STABLE/RBRICS_MINORITY_MP2_DIM16/plots/training_val_loss_plots/Lipophilicity_train_

In [3]:
# Training and Validation Losses plot
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import LogLocator

arch_colors = {
    "GAT": "tab:blue",
    "GCN": "tab:orange",
    "GIN": "tab:green",
    "SAGE": "tab:red"
}

# Define line styles for different GNN LRs
lr_line_styles = {
    "0001": "-",     # Solid line
    "001": "--",      # Dashed line
    "01": ":",        # Dotted line
    "005": "-.",      # Dash-dot line (add more if needed)
}

# Set global matplotlib styles for professional plots
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
    "axes.titlesize": 14,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 10,  # Slightly smaller for more entries
    "axes.linewidth": 1.2,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "lines.linewidth": 2.0,
})

# Ensure output directory exists
output_dir = f"{folder_name}/plots/training_val_loss_plots"
os.makedirs(output_dir, exist_ok=True)

# Get dataset names
datasets = list(training_losses.keys())

# Assuming there's only one EXPL LR value
expl_lr = expl_lr_values[0]  # Use the first (and only) expl_lr

for dataset in datasets:
    # Create figure with 2 rows (train/val) and 1 column (this dataset)
    fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=False, sharey=False)
    
    # Track all unique (arch, gnn_lr) combinations we plot
    unique_combinations = set()
    
    # Process training loss
    ax_train = axes[0]
    train_dict = training_losses[dataset]
    
    for arch in architectures:
        for gnn_lr in gnn_lr_values:
            # Skip condition for GIN validation
            skip_condition = (arch == "GIN" and gnn_lr == "01")
            
            if skip_condition:
                continue
                
            if (arch in train_dict and 
                expl_lr in train_dict[arch] and 
                gnn_lr in train_dict[arch][expl_lr]):
                
                runs = train_dict[arch][expl_lr][gnn_lr]
                min_len = min(len(r) for r in runs)
                losses = np.array([r[:min_len] for r in runs])
                mean_loss = losses.mean(axis=0)
                std_loss = losses.std(axis=0)
                epochs = np.arange(min_len)
                
                # Create unique combination identifier
                combo_id = f"{arch}-GNN LR 0.{gnn_lr}"
                unique_combinations.add(combo_id)
                
                # Plot with architecture color and LR line style
                ax_train.plot(epochs, mean_loss, 
                             label=combo_id,
                             color=arch_colors[arch],
                             linestyle=lr_line_styles.get(gnn_lr, "-"))
                ax_train.fill_between(epochs, 
                                     mean_loss - std_loss, 
                                     mean_loss + std_loss,
                                     color=arch_colors[arch], 
                                     alpha=0.15)
    
    # Configure training plot
    ax_train.set_yscale('log')
    ax_train.grid(True, linestyle='--', alpha=0.7)
    ax_train.set_title(f"{dataset} - Training Loss", fontsize=14)
    ax_train.set_ylabel("Loss (log scale)", fontsize=12)

    # Process validation loss
    ax_val = axes[1]
    val_dict = validation_losses[dataset]
    
    for arch in architectures:
        for gnn_lr in gnn_lr_values:
            # Skip condition for GIN validation
            skip_condition = (arch == "GIN" and gnn_lr == "01")
            if skip_condition:
                continue
                
            if (arch in val_dict and 
                expl_lr in val_dict[arch] and 
                gnn_lr in val_dict[arch][expl_lr]):
                
                runs = val_dict[arch][expl_lr][gnn_lr]
                min_len = min(len(r) for r in runs)
                losses = np.array([r[:min_len] for r in runs])
                mean_loss = losses.mean(axis=0)
                std_loss = losses.std(axis=0)
                epochs = np.arange(min_len)
                
                # Create unique combination identifier
                combo_id = f"{arch}-GNN LR 0.{gnn_lr}"
                unique_combinations.add(combo_id)
                
                # Plot with same style as training
                ax_val.plot(epochs, mean_loss, 
                           label=combo_id,
                           color=arch_colors[arch],
                           linestyle=lr_line_styles.get(gnn_lr, "-"))
                ax_val.fill_between(epochs, 
                                   mean_loss - std_loss, 
                                   mean_loss + std_loss,
                                   color=arch_colors[arch], 
                                   alpha=0.15)
    
    # Configure validation plot
    ax_val.set_yscale('log')
    ax_val.grid(True, linestyle='--', alpha=0.7)
    ax_val.set_title(f"{dataset} - Validation Loss", fontsize=14)
    ax_val.set_xlabel("Epoch", fontsize=12)
    ax_val.set_ylabel("Loss (log scale)", fontsize=12)
    
    # Create unified legend
    # Sort combinations for consistent ordering
    sorted_combinations = sorted(unique_combinations)
    
    # Create custom legend handles
    legend_handles = []
    for combo in sorted_combinations:
        arch = combo.split("-")[0]
        gnn_lr = combo.split(" ")[-1]
        linestyle = lr_line_styles.get(gnn_lr.split(".")[-1], "-")
        
        legend_handles.append(
            Line2D([0], [0], 
                   color=arch_colors[arch],
                   linestyle=linestyle,
                   label=combo)
        )
    
    # Place legend below the plots
    fig.legend(handles=legend_handles, 
               loc="lower center", 
               bbox_to_anchor=(0.5, -0.1),
               ncol=3,  # Adjust columns based on number of entries
               fontsize=10,
               frameon=True)

    # Set main title
    fig.suptitle(f"Training and Validation Losses (Expl LR = 0.{expl_lr})", 
                 fontsize=16, weight='bold')

    # Adjust layout and save
    fig.tight_layout(rect=[0, 0.05, 1, 0.95])  # Leave space at bottom for legend
    out_file = f"{output_dir}/{dataset}_expl_lr_{expl_lr}_losses.png"
    fig.savefig(out_file, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"✅ Saved plot: {out_file}")

✅ Saved plot: ../STABLE/RBRICS_MINORITY_MP2_DIM16/plots/training_val_loss_plots/Mutagenicity_expl_lr_01_losses.png
✅ Saved plot: ../STABLE/RBRICS_MINORITY_MP2_DIM16/plots/training_val_loss_plots/hERG_expl_lr_01_losses.png
✅ Saved plot: ../STABLE/RBRICS_MINORITY_MP2_DIM16/plots/training_val_loss_plots/BBBP_expl_lr_01_losses.png
✅ Saved plot: ../STABLE/RBRICS_MINORITY_MP2_DIM16/plots/training_val_loss_plots/esol_expl_lr_01_losses.png
✅ Saved plot: ../STABLE/RBRICS_MINORITY_MP2_DIM16/plots/training_val_loss_plots/Lipophilicity_expl_lr_01_losses.png
✅ Saved plot: ../STABLE/RBRICS_MINORITY_MP2_DIM16/plots/training_val_loss_plots/Benzene_expl_lr_01_losses.png
✅ Saved plot: ../STABLE/RBRICS_MINORITY_MP2_DIM16/plots/training_val_loss_plots/Alkane_Carbonyl_expl_lr_01_losses.png
✅ Saved plot: ../STABLE/RBRICS_MINORITY_MP2_DIM16/plots/training_val_loss_plots/Fluoride_Carbonyl_expl_lr_01_losses.png


## Showing importance to impact. Saves Top 10 motifs


In [3]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# Fixed learning rate values
# expl_lr_values = ["0001", "001", "01"]
# gnn_lr_values = ["0001", "001", "01"]
expl_lr_values = ["01"]
gnn_lr_values = ["0001", "001"]

# Create base output directory
os.makedirs(f"{folder_name}/plots/importance_impact", exist_ok=True)

# Set seaborn style
sns.set_style("whitegrid")
sns.set_context("talk", font_scale=0.8)

# Constants
unique_datasets = sorted(root_dirs_dict.keys())
unique_types = ["RBRICS"]
n_rows = len(unique_datasets)
n_cols = len(architectures)
palette = "Set2"

for model_type in unique_types:
    for EXPL_LR in expl_lr_values:
        for GNN_LR in gnn_lr_values:

            # Create output directories for this LR pair
            output_csv_dir = f"{folder_name}/plots/motif_csvs/{model_type}-EXPLLR0.{EXPL_LR}-GNNLR0.{GNN_LR}"
            os.makedirs(output_csv_dir, exist_ok=True)

            # Create figure once per LR pair
            fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 7, n_rows * 5), squeeze=False)
            plt.subplots_adjust(left=0.15, right=0.95, top=0.9, bottom=0.1, wspace=0.35, hspace=0.4)

            for d_idx, dataset in enumerate(unique_datasets):
                for a_idx, arch in enumerate(sorted(architectures)):
                    ax = axes[d_idx, a_idx]

                    # Access data
                    data = (
                        plotting_data.get(arch, {})
                                     .get(dataset, {})
                                     .get(model_type, {})
                                     .get(EXPL_LR, {})
                                     .get(GNN_LR, pd.DataFrame())
                    )

                    if data.empty:
                        ax.set_visible(False)
                        continue

                    # Filter out UNK motifs
                    data = data[data["motif"] != "UNK"]

                    # Find graph with maximum logit difference per motif
                    idx_max_logit_diff = data.groupby("motif")["logit_diff"].idxmax()
                    graph_max_logit_diff = data.loc[idx_max_logit_diff, ["motif", "graph_str"]].rename(
                        columns={"graph_str": "max_logit_diff_graph"}
                    )

                    # Deduplicate data for motif-graph frequency counting
                    dedup_counts = data.drop_duplicates(subset=["motif", "graph_id"])
                    motif_frequency = dedup_counts.groupby("motif")["graph_id"].nunique().reset_index().rename(
                        columns={"graph_id": "frequency"}
                    )

                    # Compute aggregated statistics
                    motif_stats = data.groupby("motif").agg(
                        avg_sigmoid_importance=("sigmoid_importance", "mean"),
                        sigmoid_bin=("sigmoid_bin", "first"),
                        median_logit_diff=("logit_diff", "median"),
                        mode_sign=("diff_sign", lambda x: pd.Series.mode(x).iloc[0] if not pd.Series.mode(x).empty else np.nan),
                        mode_logit_diff=("logit_diff", lambda x: pd.Series.mode(x).iloc[0] if not pd.Series.mode(x).empty else np.nan),
                        avg_logit_diff=("logit_diff", "mean"),
                        max_logit_diff=("logit_diff", "max"),
                        duplicate_counts=("motif", "count")
                    ).reset_index()

                    # Merge frequency and max-logit-graph info
                    motif_stats = (
                        motif_stats
                        .merge(motif_frequency, on="motif", how="left")
                        .merge(graph_max_logit_diff, on="motif", how="left")
                    )

                    # Sort and extract top and bottom 10 motifs
                    top_10_motifs = motif_stats.sort_values(by="avg_sigmoid_importance", ascending=False).head(10)
                    bottom_10_motifs = motif_stats.sort_values(by="avg_sigmoid_importance", ascending=True).head(10)

                    # Save to CSV
                    top_10_path = f"{output_csv_dir}/{dataset}_{arch}_top10.csv"
                    bottom_10_path = f"{output_csv_dir}/{dataset}_{arch}_bottom10.csv"
                    top_10_motifs.to_csv(top_10_path, index=False)
                    bottom_10_motifs.to_csv(bottom_10_path, index=False)

                    # Plotting
                    counts = data['sigmoid_bin'].value_counts().sort_index()
                    counts_percentage = counts / counts.sum() if counts.sum() != 0 else pd.Series()

                    sns.boxplot(
                        ax=ax,
                        x='sigmoid_bin',
                        y='avg_logit_diff',  # Change to 'median_logit_diff' or 'mode_logit_diff' if needed
                        data=motif_stats,
                        color='tab:blue',
                        width=0.6,
                        linewidth=1.5
                    )

                    if not counts_percentage.empty:
                        ax.bar(
                            x=np.arange(len(counts_percentage)),
                            height=-counts_percentage.values,
                            color='tab:orange',
                            alpha=0.4,
                            width=0.5,
                            align='center'
                        )

                    ax.axhline(0, color='black', linewidth=1, linestyle='--')
                    ax.set_ylim(-0.6, 0.6)
                    ax.set_yticks([-0.5, -0.25, 0, 0.25, 0.5, 1.0])
                    ax.set_yticklabels(['50%', '25%', '0', '0.25', '0.5', '1.0'])

                    ax.set_xlabel('Normalized Motif Importance', fontsize=12)
                    if a_idx == 0:
                        ax.set_ylabel('Freq | Abs Prob Diff', fontsize=12)
                    else:
                        ax.set_ylabel('')
                    
                    ax.tick_params(axis='x', rotation=45)

            # Row labels (datasets)
            for d_idx, dataset in enumerate(unique_datasets):
                pos = axes[d_idx][0].get_position()
                fig.text(pos.x0 - 0.05, pos.y0 + pos.height / 2, dataset,
                         ha='right', va='center', rotation=90, fontsize=18, fontweight='bold')

            # Column labels (architectures)
            for a_idx, arch in enumerate(sorted(architectures)):
                pos = axes[0][a_idx].get_position()
                fig.text(pos.x0 + pos.width / 2, pos.y1 + 0.03, arch,
                         ha='center', va='bottom', fontsize=18, fontweight='bold')

            # Save figure
            output_plot_path = f"{folder_name}/plots/importance_impact/{model_type}-EXPLLR0.{EXPL_LR}-GNNLR0.{GNN_LR}_combined_comparison.png"
            plt.savefig(output_plot_path, bbox_inches='tight')
            plt.close()

print(f"✅ Saved all plots and CSVs to {folder_name}/plots/")


✅ Saved all plots and CSVs to ../STABLE/RBRICS_MINORITY_MP2_DIM16/plots/


In [8]:

# Per dataset

import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from itertools import product

# Fixed learning rate values
# expl_lr_values = ["0001", "001", "01"]
# gnn_lr_values = ["0001", "001", "01"]
expl_lr_values = ["01"]
gnn_lr_values = ["0001", "001"]
lr_pairs = list(product(expl_lr_values, gnn_lr_values))

# Create base output directory
os.makedirs(f"{folder_name}/plots", exist_ok=True)

# Set seaborn style
sns.set_style("whitegrid")
sns.set_context("talk", font_scale=0.8)

# Constants
unique_types = ["RBRICS"]
palette = "Set2"

for model_type in unique_types:
    for dataset in sorted(root_dirs_dict.keys()):
        n_rows = len(sorted(architectures))
        n_cols = len(lr_pairs)

        # Create mega-plot figure for this dataset
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 6, n_rows * 5), squeeze=False)
        plt.subplots_adjust(left=0.1, right=0.95, top=0.9, bottom=0.1, wspace=0.3, hspace=0.4)

        for row_idx, arch in enumerate(sorted(architectures)):
            for col_idx, (EXPL_LR, GNN_LR) in enumerate(lr_pairs):
                ax = axes[row_idx, col_idx]

                # Access data
                data = (
                    plotting_data.get(arch, {})
                                 .get(dataset, {})
                                 .get(model_type, {})
                                 .get(EXPL_LR, {})
                                 .get(GNN_LR, pd.DataFrame())
                )

                if data.empty or data.shape[0] == 0:
                    ax.set_visible(False)
                    continue

                # Filter out UNK motifs
                data = data[data["motif"] != "UNK"]

                # Find graph with maximum logit difference per motif
                idx_max_logit_diff = data.groupby("motif")["logit_diff"].idxmax()
                graph_max_logit_diff = data.loc[idx_max_logit_diff, ["motif", "graph_str"]].rename(
                    columns={"graph_str": "max_logit_diff_graph"}
                )

                # Deduplicate data for motif-graph frequency counting
                dedup_counts = data.drop_duplicates(subset=["motif", "graph_id"])
                motif_frequency = dedup_counts.groupby("motif")["graph_id"].nunique().reset_index().rename(
                    columns={"graph_id": "frequency"}
                )

                # Compute aggregated statistics
                motif_stats = data.groupby("motif").agg(
                    avg_sigmoid_importance=("sigmoid_importance", "mean"),
                    sigmoid_bin=("sigmoid_bin", "first"),
                    median_logit_diff=("logit_diff", "median"),
                    mode_sign=("diff_sign", lambda x: pd.Series.mode(x).iloc[0] if not pd.Series.mode(x).empty else np.nan),
                    mode_logit_diff=("logit_diff", lambda x: pd.Series.mode(x).iloc[0] if not pd.Series.mode(x).empty else np.nan),
                    avg_logit_diff=("logit_diff", "mean"),
                    max_logit_diff=("logit_diff", "max"),
                    duplicate_counts=("motif", "count")
                ).reset_index()

                # Merge frequency and max-logit-graph info
                motif_stats = (
                    motif_stats
                    .merge(motif_frequency, on="motif", how="left")
                    .merge(graph_max_logit_diff, on="motif", how="left")
                )

                # Sort and extract top and bottom 10 motifs
                # output_csv_dir = f"{folder_name}/plots/motif_csvs/{model_type}-EXPLLR{EXPL_LR}-GNNLR{GNN_LR}"
                # os.makedirs(output_csv_dir, exist_ok=True)
                # top_10_path = f"{output_csv_dir}/{dataset}_{arch}_top10.csv"
                # bottom_10_path = f"{output_csv_dir}/{dataset}_{arch}_bottom10.csv"
                # motif_stats.sort_values(by="avg_sigmoid_importance", ascending=False).head(10).to_csv(top_10_path, index=False)
                # motif_stats.sort_values(by="avg_sigmoid_importance", ascending=True).head(10).to_csv(bottom_10_path, index=False)

                # Plotting
                counts = data['sigmoid_bin'].value_counts().sort_index()
                counts_percentage = counts / counts.sum() if counts.sum() != 0 else pd.Series()

                sns.boxplot(
                    ax=ax,
                    x='sigmoid_bin',
                    y='avg_logit_diff',  # Change to 'median_logit_diff' or 'mode_logit_diff' if preferred
                    data=motif_stats,
                    color='tab:blue',
                    width=0.6,
                    linewidth=1.5
                )

                if not counts_percentage.empty:
                    ax.bar(
                        x=np.arange(len(counts_percentage)),
                        height=-counts_percentage.values,
                        color='tab:orange',
                        alpha=0.4,
                        width=0.5,
                        align='center'
                    )

                ax.axhline(0, color='black', linewidth=1, linestyle='--')
                ax.set_ylim(-0.6, 0.6)
                ax.set_yticks([-0.5, -0.25, 0, 0.25, 0.5, 1.0])
                ax.set_yticklabels(['50%', '25%', '0', '0.25', '0.5', '1.0'])

                # Titles and labels
                if row_idx == 0:
                    ax.set_title(f"EXPLLR{EXPL_LR} - GNNLR{GNN_LR}", fontsize=14, fontweight='bold')
                if col_idx == 0:
                    ax.set_ylabel(f"{arch}", fontsize=14, fontweight='bold')
                ax.set_xlabel('Sigmoid Bin', fontsize=10)
                ax.tick_params(axis='x', rotation=45)

        # Save mega-plot for this dataset
        output_plot_path = f"{folder_name}/plots/{dataset}_{model_type}_mega_plot.png"
        plt.suptitle(f"Mega Plot: {dataset} - {model_type}", fontsize=18, fontweight='bold')
        plt.savefig(output_plot_path, bbox_inches='tight')
        plt.close()

print(f"✅ Saved all mega-plots and CSVs to {folder_name}/plots/")


✅ Saved all mega-plots and CSVs to ../RBRICS_MINORITY_MP2_DIM16_CORRECTED/plots/


##  save performance prediction and importance impact correlation to table

In [8]:
# import os
# import re
# import json
# import pandas as pd
# import numpy as np
# from collections import defaultdict
# from scipy.stats import spearmanr, kendalltau, pearsonr

# expl_lr_values = ["0001", "001", "01"]
# gnn_lr_values = ["0001", "001", "01"]

# VANILLA_FOLDER_NAME = "../Vanilla_CORRECTED"
# model_type = "RBRICS"

# root_dirs_dict = {
#     "Mutagenicity": (f"{folder_name}/MOSE_BC", f"{VANILLA_FOLDER_NAME}/Vanilla_BC"),
#     "hERG": (f"{folder_name}/MOSE_BC", f"{VANILLA_FOLDER_NAME}/Vanilla_BC"),
#     "BBBP": (f"{folder_name}/MOSE_BC", f"{VANILLA_FOLDER_NAME}/Vanilla_BC"),
#     "esol": (f"{folder_name}/MOSE_Reg", f"{VANILLA_FOLDER_NAME}/Vanilla_Reg"),
#     "Lipophilicity": (f"{folder_name}/MOSE_Reg", f"{VANILLA_FOLDER_NAME}/Vanilla_Reg"),
#     "Benzene": (f"{folder_name}/MOSE_BC", f"{VANILLA_FOLDER_NAME}/Vanilla_BC"),
#     "Alkane_Carbonyl": (f"{folder_name}/MOSE_BC", f"{VANILLA_FOLDER_NAME}/Vanilla_BC"),
#     "Fluoride_Carbonyl": (f"{folder_name}/MOSE_BC", f"{VANILLA_FOLDER_NAME}/Vanilla_BC"),
# }

# architectures = {"GAT", "GCN", "GIN", "SAGE"}
# all_rows = []

# for EXPL_LR in expl_lr_values:
#     for GNN_LR in gnn_lr_values:
#         print(f"\n🔍 Processing EXPLLR=0.{EXPL_LR}, GNNLR=0.{GNN_LR}...")
#         performance_data = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

#         # Regex for MOSE and Vanilla folders
#         folder_pattern = re.compile(
#             r"^EXPT-(?P<expt_id>\d+[A-Z]*)-"
#             r"(?P<dataset>[\w_]+)-"
#             r"SEED-(?P<seed>\d+)-"
#             r"FOLD-(?P<fold>\d+)-"
#             r"(?P<arch>\w+)-"
#             fr"EXPLLR0\.{EXPL_LR}-GNNLR0\.{GNN_LR}-SingleChannel-RBRICS$"
#         )

#         folder_pattern_vanilla = re.compile(
#             r"^EXPT-(?P<expt_id>\d+[A-Z]*)-"
#             r"(?P<dataset>[\w_]+)-"
#             r"SEED-(?P<seed>\d+)-"
#             r"FOLD-(?P<fold>\d+)-"
#             r"(?P<arch>\w+)-"
#             fr"MP2-DIM16-GNNLR0\.{GNN_LR}-Vanilla-None$"
#         )

#         for dataset_name, (expl_root, van_root) in root_dirs_dict.items():
#             # Index Vanilla folders
#             vanilla_lookup = {}
#             for vfolder in os.listdir(van_root):
#                 m = folder_pattern_vanilla.match(vfolder)
#                 if not m:
#                     continue
#                 key = (m["dataset"], m["seed"], m["fold"], m["arch"])
#                 vanilla_lookup[key] = vfolder

#             # Walk through MOSE folders
#             for efolder in os.listdir(expl_root):
#                 m = folder_pattern.match(efolder)
#                 if not m:
#                     continue

#                 dataset, seed, fold, arch = (
#                     m["dataset"],
#                     m["seed"],
#                     m["fold"],
#                     m["arch"],
#                 )

#                 if arch not in architectures:
#                     continue

#                 key = (dataset, seed, fold, arch)
#                 vfolder = vanilla_lookup.get(key)
#                 if not vfolder:
#                     print(f"⚠️ Vanilla folder missing for {key}")
#                     continue

#                 mose_file = os.path.join(expl_root, efolder, f"{dataset}_classification_result.json")
#                 vanilla_file = os.path.join(van_root, vfolder, f"{dataset}_classification_result.json")

#                 files = {"mose": mose_file, "vanilla": vanilla_file}
#                 missing = [name for name, path in files.items() if not os.path.exists(path)]
#                 if missing:
#                     print(f"⚠️ Missing files for run {efolder}:")
#                     for name in missing:
#                         print(f"    {name}: {files[name]}")
#                     continue

#                 with open(mose_file) as f:
#                     perf_mose = json.load(f)
#                 with open(vanilla_file) as f:
#                     perf_vanilla = json.load(f)

#                 metric_suffix = "rocauc" if dataset in {"BBBP", "Mutagenicity", "hERG", "Alkane_Carbonyl", "Benzene", "Fluoride_Carbonyl", "tox21"} else "rmse"

#                 for split in ["train", "validation", "test"]:
#                     mose_metric = perf_mose.get(f"Trained_explainations_{split}_{metric_suffix}", None)
#                     van_metric = perf_vanilla.get(f"Trained_explainations_{split}_{metric_suffix}", None)
#                     performance_data[dataset][arch][f"Mose {split.capitalize()}"].append(mose_metric)
#                     performance_data[dataset][arch][f"Vanilla {split.capitalize()}"].append(van_metric)

#         # Flatten nested dict
#         for dataset, arch_dict in performance_data.items():
#             for arch, metrics in arch_dict.items():
#                 row = {
#                     "Dataset": dataset,
#                     "Architecture": arch,
#                     "EXPL_LR": f"0.{EXPL_LR}",
#                     "GNN_LR": f"0.{GNN_LR}"
#                 }

#                 for metric_name, values in metrics.items():
#                     if values:
#                         mean_val = round(np.nanmean(values), 4)
#                         std_val = round(pd.Series(values).std(), 4)
#                         row[f"{metric_name} Mean"] = mean_val
#                         row[f"{metric_name} Std"] = std_val
#                     else:
#                         row[f"{metric_name} Mean"] = None
#                         row[f"{metric_name} Std"] = None
                        

#                 # Access data
#                 data = (
#                     plotting_data.get(arch, {})
#                                  .get(dataset, {})
#                                  .get(model_type, {})
#                                  .get(EXPL_LR, {})
#                                  .get(GNN_LR, pd.DataFrame())
#                 )


#                 # Filter out UNK motifs
#                 data = data[data["motif"] != "UNK"]


#                 # Compute aggregated statistics
#                 data = data.groupby("motif").agg(
#                     avg_sigmoid_importance=("sigmoid_importance", "mean"),
#                     median_logit_diff=("logit_diff", "median"),
#                     mode_sign=("diff_sign", lambda x: pd.Series.mode(x).iloc[0] if not pd.Series.mode(x).empty else np.nan),
#                     mode_logit_diff=("logit_diff", lambda x: pd.Series.mode(x).iloc[0] if not pd.Series.mode(x).empty else np.nan),
#                     avg_logit_diff=("logit_diff", "mean"),
#                     max_logit_diff=("logit_diff", "max")
#                 ).reset_index()
                    
#                 if not data.empty and 'avg_sigmoid_importance' in data and 'avg_logit_diff' in data:
#                     sig_imp = data['avg_sigmoid_importance']
#                     logit_diff = data['avg_logit_diff']

#                     if len(sig_imp.dropna()) >= 2 and len(logit_diff.dropna()) >= 2:
#                         row['Spearman Corr'] = round(spearmanr(sig_imp, logit_diff, nan_policy='omit')[0], 4)
#                         row['Kendall Corr'] = round(kendalltau(sig_imp, logit_diff, nan_policy='omit')[0], 4)
#                         row['Pearson Corr'] = round(pearsonr(sig_imp, logit_diff)[0], 4)
#                     else:
#                         row['Spearman Corr'] = row['Kendall Corr'] = row['Pearson Corr'] = None
#                 else:
#                     print(f"⚠️ No valid data for correlation: {dataset}-{arch} EXPLLR0.{EXPL_LR}-GNNLR0.{GNN_LR}")
#                     row['Spearman Corr'] = row['Kendall Corr'] = row['Pearson Corr'] = None

#                 all_rows.append(row)

# # Combine all rows into one DataFrame
# performance_df = pd.DataFrame(all_rows)
# performance_df = performance_df.sort_values(by=["Dataset", "Architecture", "EXPL_LR", "GNN_LR"])

# # Save big combined CSV
# output_csv_path = f"{folder_name}/performance_summary_all_lrs.csv"
# os.makedirs(folder_name, exist_ok=True)
# performance_df.to_csv(output_csv_path, index=False)
# print(f"✅ Combined performance summary with correlations saved to {output_csv_path}")



🔍 Processing EXPLLR=0.0001, GNNLR=0.0001...
⚠️ Vanilla folder missing for ('BBBP', '0', '2', 'SAGE')
⚠️ Vanilla folder missing for ('hERG', '0', '2', 'GIN')
⚠️ Missing files for run EXPT-14BC-Alkane_Carbonyl-SEED-0-FOLD-4-GIN-EXPLLR0.0001-GNNLR0.0001-SingleChannel-RBRICS:
    mose: ../RBRICS_MINORITY_MP2_DIM16_CORRECTED/MOSE_BC/EXPT-14BC-Alkane_Carbonyl-SEED-0-FOLD-4-GIN-EXPLLR0.0001-GNNLR0.0001-SingleChannel-RBRICS/Alkane_Carbonyl_classification_result.json
⚠️ Vanilla folder missing for ('Benzene', '0', '2', 'GIN')
⚠️ Vanilla folder missing for ('Fluoride_Carbonyl', '0', '2', 'SAGE')
⚠️ Missing files for run EXPT-14BC-Benzene-SEED-0-FOLD-1-GCN-EXPLLR0.0001-GNNLR0.0001-SingleChannel-RBRICS:
    mose: ../RBRICS_MINORITY_MP2_DIM16_CORRECTED/MOSE_BC/EXPT-14BC-Benzene-SEED-0-FOLD-1-GCN-EXPLLR0.0001-GNNLR0.0001-SingleChannel-RBRICS/Benzene_classification_result.json
⚠️ Vanilla folder missing for ('Benzene', '0', '2', 'SAGE')
⚠️ Missing files for run EXPT-14BC-hERG-SEED-0-FOLD-4-GCN-EXPLLR0

In [6]:
from sklearn.mixture import GaussianMixture
import numpy as np
from scipy.stats import spearmanr, kendalltau, pearsonr

def estimate_modality_score(values, max_components=4):
    """Returns a modality score: high if bimodal, low if unimodal or too many modes."""
    if len(values) < 10:
        return 0.0  # Too few points to infer
    values = values.dropna().values.reshape(-1, 1)

    bics = []
    for k in range(1, max_components + 1):
        gmm = GaussianMixture(n_components=k, random_state=0)
        gmm.fit(values)
        bics.append(gmm.bic(values))

    best_k = np.argmin(bics) + 1

    if best_k == 1:
        return 0.0  # unimodal bad explainer
    else:  
        return 1.0

In [7]:
import os
import re
import json
import pandas as pd
import numpy as np
from collections import defaultdict
from scipy.stats import spearmanr, kendalltau, pearsonr

# expl_lr_values = ["0001", "001", "01"]
# gnn_lr_values = ["0001", "001", "01"]
expl_lr_values = ["01"]
gnn_lr_values = ["0001", "001"]

VANILLA_FOLDER_NAME = "../Vanilla_CORRECTED"
# VANILLA_FOLDER_NAME = "../STABLE"
missing_data_records = []

root_dirs_dict = {
    "Mutagenicity": (f"{folder_name}/MOSE_BC", f"{VANILLA_FOLDER_NAME}/Vanilla_BC"),
    "hERG": (f"{folder_name}/MOSE_BC", f"{VANILLA_FOLDER_NAME}/Vanilla_BC"),
    "BBBP": (f"{folder_name}/MOSE_BC", f"{VANILLA_FOLDER_NAME}/Vanilla_BC"),
    "esol": (f"{folder_name}/MOSE_Reg", f"{VANILLA_FOLDER_NAME}/Vanilla_Reg"),
    "Lipophilicity": (f"{folder_name}/MOSE_Reg", f"{VANILLA_FOLDER_NAME}/Vanilla_Reg"),
    "Benzene": (f"{folder_name}/MOSE_BC", f"{VANILLA_FOLDER_NAME}/Vanilla_BC"),
    "Alkane_Carbonyl": (f"{folder_name}/MOSE_BC", f"{VANILLA_FOLDER_NAME}/Vanilla_BC"),
    "Fluoride_Carbonyl": (f"{folder_name}/MOSE_BC", f"{VANILLA_FOLDER_NAME}/Vanilla_BC"),
}

architectures = {"GAT", "GCN", "GIN", "SAGE"}
valid_folds = ["0", "1", "2", "3", "4"]
all_rows = []
vanilla_data = defaultdict(lambda: defaultdict(list))

# === Exhaustive check for Vanilla folders & files ===
for dataset_name, (_, van_root) in root_dirs_dict.items():
    if not os.path.exists(van_root):
        missing_data_records.append({
            "Dataset": dataset_name,
            "Architecture": "ALL",
            "Fold": "ALL",
            "Type": "Vanilla",
            "EXPL_LR": "N/A",
            "GNN_LR": "ALL",
            "Missing Files": f"Vanilla root folder missing: {van_root}"
        })
        print(f"⚠️ Vanilla root folder missing: {van_root}")
        continue

    for arch in architectures:
        for fold in valid_folds:
            for GNN_LR in gnn_lr_values:
                folder_pattern_vanilla = re.compile(
                    fr"^EXPT-(?P<expt_id>\d+[A-Z]*)-{dataset_name}-"
                    r"SEED-(?P<seed>\d+)-"
                    fr"FOLD-{fold}-"
                    fr"{arch}-"
                    fr"MP2-DIM16-GNNLR0\.{GNN_LR}-Vanilla-None$"
                )

                folder_found = False
                for vfolder in os.listdir(van_root):
                    if folder_pattern_vanilla.fullmatch(vfolder):
                        folder_found = True
                        vanilla_file = os.path.join(van_root, vfolder, f"{dataset_name}_classification_result.json")

                        if not os.path.exists(vanilla_file):
                            missing_data_records.append({
                                "Dataset": dataset_name,
                                "Architecture": arch,
                                "Fold": fold,
                                "Type": "Vanilla",
                                "EXPL_LR": "N/A",
                                "GNN_LR": f"0.{GNN_LR}",
                                "Missing Files": f"Missing Vanilla JSON file: {vanilla_file}"
                            })
                            print(f"⚠️ Missing Vanilla file: {vanilla_file}")
                        else:
                            # Load Vanilla results
                            try:
                                with open(vanilla_file) as f:
                                    perf_vanilla = json.load(f)
                                metric_suffix = "rocauc" if dataset_name.lower() in {
                                    "bbbp", "mutagenicity", "herg",
                                    "alkane_carbonyl", "benzene",
                                    "fluoride_carbonyl", "tox21"
                                } else "rmse"
                                for split in ["train", "validation", "test"]:
                                    van_metric = perf_vanilla.get(f"Trained_explainations_{split}_{metric_suffix}", None)
                                    vanilla_data[(dataset_name, arch, GNN_LR)][f"Vanilla {split.capitalize()}"] += [van_metric]
                            except Exception as e:
                                missing_data_records.append({
                                    "Dataset": dataset_name,
                                    "Architecture": arch,
                                    "Fold": fold,
                                    "Type": "Vanilla",
                                    "EXPL_LR": "N/A",
                                    "GNN_LR": f"0.{GNN_LR}",
                                    "Missing Files": f"Error reading Vanilla JSON: {e}"
                                })
                                print(f"⚠️ Error reading Vanilla JSON: {vanilla_file} ({e})")
                        break

                if not folder_found:
                    missing_data_records.append({
                        "Dataset": dataset_name,
                        "Architecture": arch,
                        "Fold": fold,
                        "Type": "Vanilla",
                        "EXPL_LR": "N/A",
                        "GNN_LR": f"0.{GNN_LR}",
                        "Missing Files": "Vanilla folder matching pattern not found"
                    })
                    print(f"⚠️ Missing Vanilla folder for: Dataset={dataset_name}, Arch={arch}, Fold={fold}, GNN_LR=0.{GNN_LR}")

# Process MOSE results and compute correlations
for EXPL_LR in expl_lr_values:
    for GNN_LR in gnn_lr_values:
        print(f"\n🔍 Processing EXPLLR=0.{EXPL_LR}, GNNLR=0.{GNN_LR}...")
        performance_data = defaultdict(lambda: defaultdict(list))

        for dataset_name, (expl_root, _) in root_dirs_dict.items():
            for efolder in os.listdir(expl_root):
                # Regex for MOSE and Vanilla folders
                folder_pattern = re.compile(
                    fr"^EXPT-(?P<expt_id>\d+[A-Z]*)-{dataset_name}-"
                    r"SEED-(?P<seed>\d+)-"
                    r"FOLD-(?P<fold>\d+)-"
                    r"(?P<arch>\w+)-"
                    fr"EXPLLR0\.{EXPL_LR}-GNNLR0\.{GNN_LR}-SingleChannel-RBRICS$"
                )
                m = folder_pattern.match(efolder)
                if not m:
                    continue
                arch = m["arch"]
                mose_file = os.path.join(expl_root, efolder, f"{dataset_name}_classification_result.json")
                if not os.path.exists(mose_file):
                    continue
                with open(mose_file) as f:
                    perf_mose = json.load(f)
                metric_suffix = "rocauc" if dataset_name.lower() in {"bbbp", "mutagenicity", "herg", "alkane_carbonyl", "benzene", "fluoride_carbonyl", "tox21"} else "rmse"
                for split in ["train", "validation", "test"]:
                    mose_metric = perf_mose.get(f"Trained_explainations_{split}_{metric_suffix}", None)
                    performance_data[(dataset_name, arch, GNN_LR)][f"Mose {split.capitalize()}"] += [mose_metric]

        # Compile rows for this LR pair
        for (dataset, arch, gnn_lr), metrics in performance_data.items():
            row = {
                "Dataset": dataset,
                "Architecture": arch,
                "EXPL_LR": f"0.{EXPL_LR}",
                "GNN_LR": f"0.{gnn_lr}"
            }
            # Add MOSE metrics
            for metric_name, values in metrics.items():
                if values:
                    row[f"{metric_name} Mean"] = round(np.nanmean(values), 4)
                    row[f"{metric_name} Std"] = round(pd.Series(values).std(), 4)
                else:
                    row[f"{metric_name} Mean"] = None
                    row[f"{metric_name} Std"] = None
            # Add Vanilla metrics
            vanilla_metrics = vanilla_data.get((dataset, arch, gnn_lr), {})
            for metric_name, values in vanilla_metrics.items():
                if values:
                    row[f"{metric_name} Mean"] = round(np.nanmean(values), 4)
                    row[f"{metric_name} Std"] = round(pd.Series(values).std(), 4)
                else:
                    row[f"{metric_name} Mean"] = None
                    row[f"{metric_name} Std"] = None
            # Correlation metrics using all points
            data = (
                plotting_data.get(arch, {})
                             .get(dataset, {})
                             .get(model_type, {})
                             .get(EXPL_LR, {})
                             .get(GNN_LR, pd.DataFrame())
            )
            
            if not data.empty:
            
                # Compute aggregated statistics
                data = data.groupby("motif").agg(
                    avg_sigmoid_importance=("sigmoid_importance", "mean"),
                    median_logit_diff=("logit_diff", "median"),
                    mode_sign=("diff_sign", lambda x: pd.Series.mode(x).iloc[0] if not pd.Series.mode(x).empty else np.nan),
                    mode_logit_diff=("logit_diff", lambda x: pd.Series.mode(x).iloc[0] if not pd.Series.mode(x).empty else np.nan),
                    avg_logit_diff=("logit_diff", "mean"),
                    max_logit_diff=("logit_diff", "max")
                ).reset_index()
                
            # if not data.empty and 'avg_sigmoid_importance' in data and 'avg_logit_diff' in data:
            #     sig_imp = data['avg_sigmoid_importance']
            #     logit_diff = data['avg_logit_diff']
            #     if len(sig_imp.dropna()) >= 2 and len(logit_diff.dropna()) >= 2:
            #         row['Spearman Corr'] = round(spearmanr(sig_imp, logit_diff, nan_policy='omit')[0], 4)
            #         row['Kendall Corr'] = round(kendalltau(sig_imp, logit_diff, nan_policy='omit')[0], 4)
            #         row['Pearson Corr'] = round(pearsonr(sig_imp, logit_diff)[0], 4)
            #     else:
            #         row['Spearman Corr'] = row['Kendall Corr'] = row['Pearson Corr'] = None
            # else:
            #     row['Spearman Corr'] = row['Kendall Corr'] = row['Pearson Corr'] = None
                
            if not data.empty and 'avg_sigmoid_importance' in data and 'avg_logit_diff' in data:
                sig_imp = data['avg_sigmoid_importance']
                logit_diff = data['avg_logit_diff']
                motif_freq = data['motif'].map(data['motif'].value_counts())

                if len(sig_imp.dropna()) >= 10 and len(logit_diff.dropna()) >= 2:
                    # Raw correlations
                    rho_s = spearmanr(sig_imp, logit_diff, nan_policy='omit')[0]
                    rho_k = kendalltau(sig_imp, logit_diff, nan_policy='omit')[0]
                    rho_p = pearsonr(sig_imp, logit_diff.dropna())[0]

                    # Penalize based on number of near-zero importances
                    zero_ratio = (sig_imp < 0.1).sum() / len(sig_imp)
                    penalty = 1 - zero_ratio  # 0 if all near-zero, 1 if all informative

                    # Apply penalty
                    weighted_s = abs(rho_s) * penalty
                    weighted_k = abs(rho_k) * penalty
                    weighted_p = abs(rho_p) * penalty

                    row['Spearman Corr'] = round(rho_s, 4)
                    row['Kendall Corr'] = round(rho_k, 4)
                    row['Pearson Corr'] = round(rho_p, 4)

                    row['Weighted Spearman'] = round(weighted_s, 4)
                    row['Weighted Kendall'] = round(weighted_k, 4)
                    row['Weighted Pearson'] = round(weighted_p, 4)

                    row['Zero Ratio Penalty'] = round(penalty, 4)
                else:
                    row['Spearman Corr'] = row['Kendall Corr'] = row['Pearson Corr'] = None
                    row['Weighted Spearman'] = row['Weighted Kendall'] = row['Weighted Pearson'] = None
                    row['Zero Ratio Penalty'] = None
            all_rows.append(row)

# Save combined DataFrame
performance_df = pd.DataFrame(all_rows)
performance_df = performance_df.sort_values(by=["Dataset", "Architecture", "EXPL_LR", "GNN_LR"])
output_csv_path = f"{folder_name}/performance_summary_all_lrs_2.csv"
os.makedirs(folder_name, exist_ok=True)
performance_df.to_csv(output_csv_path, index=False)
print(f"✅ Saved performance summary with correlations to {output_csv_path}")

# Save missing data log
if missing_data_records:
    missing_df = pd.DataFrame(missing_data_records)
    missing_df.to_csv(f"{folder_name}/missing_data_report_vanilla.csv", index=False)
    print(f"⚠️ Saved missing data report to {folder_name}/missing_data_report.csv")

⚠️ Missing Vanilla folder for: Dataset=Mutagenicity, Arch=GIN, Fold=2, GNN_LR=0.0001
⚠️ Missing Vanilla folder for: Dataset=Mutagenicity, Arch=GIN, Fold=2, GNN_LR=0.001
⚠️ Missing Vanilla folder for: Dataset=Mutagenicity, Arch=SAGE, Fold=2, GNN_LR=0.0001
⚠️ Missing Vanilla folder for: Dataset=Mutagenicity, Arch=SAGE, Fold=2, GNN_LR=0.001
⚠️ Missing Vanilla folder for: Dataset=hERG, Arch=GIN, Fold=2, GNN_LR=0.0001
⚠️ Missing Vanilla folder for: Dataset=hERG, Arch=GIN, Fold=2, GNN_LR=0.001
⚠️ Missing Vanilla folder for: Dataset=hERG, Arch=SAGE, Fold=2, GNN_LR=0.0001
⚠️ Missing Vanilla folder for: Dataset=hERG, Arch=SAGE, Fold=2, GNN_LR=0.001
⚠️ Missing Vanilla folder for: Dataset=BBBP, Arch=GIN, Fold=2, GNN_LR=0.0001
⚠️ Missing Vanilla folder for: Dataset=BBBP, Arch=GIN, Fold=2, GNN_LR=0.001
⚠️ Missing Vanilla folder for: Dataset=BBBP, Arch=SAGE, Fold=2, GNN_LR=0.0001
⚠️ Missing Vanilla folder for: Dataset=BBBP, Arch=SAGE, Fold=2, GNN_LR=0.001
⚠️ Missing Vanilla folder for: Dataset=Benze

In [15]:
performance_df

,Dataset,Architecture,EXPL_LR,GNN_LR,Mose Train Mean,Mose Train Std,Mose Validation Mean,Mose Validation Std,Mose Test Mean,Mose Test Std,...,Vanilla Test Mean,Vanilla Test Std,Spearman Corr,Kendall Corr,Pearson Corr,Weighted Spearman,Weighted Kendall,Weighted Pearson,Modality Score,Freq Penalty
21,Alkane_Carbonyl,GAT,0.01,0.0001,0.9071,0.0155,0.8721,0.0131,0.8658,0.0146,...,0.8424,0.0122,0.7421,0.5777,0.7884,0.7421,0.5777,0.7884,1.0,1.0
54,Alkane_Carbonyl,GAT,0.01,0.001,0.9425,0.0079,0.9011,0.0019,0.9121,0.0001,...,0.8657,0.0112,0.5514,0.4265,0.7191,0.5514,0.4265,0.7191,1.0,1.0
23,Alkane_Carbonyl,GCN,0.01,0.0001,0.8923,0.0070,0.8690,0.0034,0.8653,0.0161,...,0.8261,0.0026,0.6174,0.4701,0.7706,0.6174,0.4701,0.7706,1.0,1.0
55,Alkane_Carbonyl,GCN,0.01,0.001,0.9443,0.0025,0.8940,0.0025,0.9070,0.0137,...,0.8637,0.0209,0.5384,0.4265,0.6913,0.5384,0.4265,0.6913,1.0,1.0
22,Alkane_Carbonyl,GIN,0.01,0.0001,0.9334,0.0004,0.9060,0.0023,0.9001,0.0145,...,0.9084,0.0213,0.4880,0.3563,0.6259,0.4880,0.3563,0.6259,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33,hERG,GCN,0.01,0.001,0.7911,0.0113,0.7671,0.0202,0.7732,0.0146,...,0.7558,0.0118,0.8967,0.7423,0.8269,0.8967,0.7423,0.8269,1.0,1.0
5,hERG,GIN,0.01,0.0001,0.8004,0.0160,0.7743,0.0107,0.7674,0.0125,...,0.7782,NaN,0.8313,0.6691,0.7813,0.8313,0.6691,0.7813,1.0,1.0
34,hERG,GIN,0.01,0.001,0.8417,0.0165,0.8042,0.0115,0.7939,0.0187,...,0.8059,NaN,0.5671,0.4071,0.6428,0.0000,0.0000,0.0000,0.0,1.0
7,hERG,SAGE,0.01,0.0001,0.7416,NaN,0.7437,NaN,0.7190,NaN,...,0.7147,NaN,0.9602,0.8443,0.8085,0.9602,0.8443,0.8085,1.0,1.0


In [ ]:
input()

In [8]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
from itertools import product

# Fixed learning rates
expl_lr_values = ["0001", "001", "01"]
gnn_lr_values = ["0001", "001", "01"]

# Columns to retain
columns_to_keep = [
    "motif", "avg_sigmoid_importance", "median_logit_diff", "mode_sign",
    "mode_logit_diff", "avg_logit_diff", "max_logit_diff", "duplicate_counts",
    "frequency", "max_logit_diff_graph"
]

# Combine motif stats and plot motif bias for all LR pairs
def process_all_lr_pairs(folder_name, model_type):
    for EXPL_LR, GNN_LR in product(expl_lr_values, gnn_lr_values):
        print(f"\n🔄 Processing EXPLLR=0.{EXPL_LR}, GNNLR=0.{GNN_LR}")
        
        csv_folder = f"{folder_name}/plots/motif_csvs/{model_type}-EXPLLR0.{EXPL_LR}-GNNLR0.{GNN_LR}"
        # input(csv_folder)
        csv_files = glob.glob(os.path.join(csv_folder, "*_*_top10.csv")) + \
                    glob.glob(os.path.join(csv_folder, "*_*_bottom10.csv"))
        all_data = []

        for file in csv_files:
            try:
                filename = os.path.basename(file).replace(".csv", "")
                parts = filename.split("_")

                if parts[-1] in {"top10", "bottom10"}:
                    label = parts[-1]
                    architecture = parts[-2]
                    dataset = "_".join(parts[:-2])
                else:
                    print(f"Skipping invalid filename format: {filename}")
                    continue

                df = pd.read_csv(file)

                if not set(columns_to_keep).issubset(df.columns):
                    print(f"Skipping {filename}: missing required columns")
                    continue

                df = df[columns_to_keep]
                float_cols = df.select_dtypes(include=['float']).columns
                df[float_cols] = df[float_cols].round(3)

                df["dataset"] = dataset
                df["architecture"] = architecture
                df["rank_type"] = label

                all_data.append(df)
            except Exception as e:
                print(f"Error processing {file}: {e}")

        if all_data:
            combined_df = pd.concat(all_data, ignore_index=True)
            combined_csv_path = f"{csv_folder}/{model_type}-EXPLLR0.{EXPL_LR}-GNNLR0.{GNN_LR}_combined_motif_stats.csv"
            combined_df.to_csv(combined_csv_path, index=False)
            print(f"✅ Saved combined_motif_stats.csv for EXPLLR=0.{EXPL_LR}, GNNLR=0.{GNN_LR}")

            plot_dir = f"{csv_folder}/plots"
            os.makedirs(plot_dir, exist_ok=True)

            csv_folder_input = f"../csv_exports"
            csv_files_input = sorted(glob.glob(os.path.join(csv_folder_input, "*_0_RBRICS.csv")))

            for f in csv_files_input:
                plot_motif_bias(f, combined_csv_path, save_dir=plot_dir)
        else:
            print(f"⚠️ No valid data found for EXPLLR=0.{EXPL_LR}, GNNLR=0.{GNN_LR}")

# Plotting function
def plot_motif_bias(csv_file, combined_csv, min_support=1, save_dir=None):
    df = pd.read_csv(csv_file)
    df['Motif Length'] = pd.to_numeric(df.get('Motif Length', 0), errors='coerce')
    df['Label'] = pd.to_numeric(df['Label'], errors='coerce')
    df = df.drop_duplicates(subset=['Graph String', 'Motif String'])
    df = df.rename(columns={'Motif String': 'motif'})

    dataset_name = os.path.basename(csv_file).split('_')[0].lower()
    combined_df = pd.read_csv(combined_csv)
    motif_meta = combined_df[combined_df['dataset'].str.lower() == dataset_name]

    df = df.merge(motif_meta[['motif', 'architecture', 'rank_type']], on='motif', how='left')
    df['rank_type'] = df['rank_type'].fillna('other')
    df['architecture'] = df['architecture'].fillna('default')

    is_regression = dataset_name in ['lipophilicity', 'esol']
    if not is_regression:
        df = df[df['Label'].isin([0, 1])]

    rank_colors = {'top10': 'tab:blue', 'bottom10': 'tab:red', 'other': 'tab:gray'}

    for arch in df['architecture'].unique():
        arch_df = df[df['architecture'] == arch]
        counts = arch_df.groupby(['motif', 'rank_type']).size().reset_index(name='count')
        motif_total = arch_df.groupby('motif').size().reset_index(name='total')
        class_ratio = arch_df.groupby('motif')['Label'].mean().reset_index(name='class1_ratio')
        merged = counts.merge(motif_total, on='motif').merge(class_ratio, on='motif')
        merged = merged[merged['total'] >= min_support]

        fig, ax = plt.subplots(figsize=(8, 6))
        plotted_labels = set()

        for _, row in merged.iterrows():
            rt = row['rank_type']
            color = rank_colors.get(rt, 'tab:gray')
            label = rt if rt not in plotted_labels else None
            facecolor = color if rt != 'bottom10' else 'none'

            ax.scatter(
                row['total'],
                row['class1_ratio'],
                facecolors=facecolor,
                edgecolors=color,
                linewidth=1,
                s=60,
                label=label
            )
            if label:
                plotted_labels.add(rt)

        ax.axhline(0.5, color='gray', linestyle='--')
        ax.set_xlabel("Motif Frequency")
        ax.set_ylabel("Class 1 Ratio" if not is_regression else "Mean Label")
        ax.set_title(f"{dataset_name.upper()} | {arch}", fontsize=12)
        ax.legend(title="Rank Type", fontsize=8, title_fontsize=9)
        plt.tight_layout()

        if save_dir:
            out_path = os.path.join(save_dir, f"{dataset_name}_{arch}_motif_bias.png")
            fig.savefig(out_path, dpi=300)
            print(f"✅ Saved: {out_path}")
            plt.close(fig)
        else:
            plt.show()

# === Run for all LR pairs ===
process_all_lr_pairs(folder_name, "RBRICS")



🔄 Processing EXPLLR=0.0001, GNNLR=0.0001
✅ Saved combined_motif_stats.csv for EXPLLR=0.0001, GNNLR=0.0001
✅ Saved: ../RBRICS_MINORITY_MP2_DIM16_CORRECTED/plots/motif_csvs/RBRICS-EXPLLR0.0001-GNNLR0.0001/plots/alkane_default_motif_bias.png
✅ Saved: ../RBRICS_MINORITY_MP2_DIM16_CORRECTED/plots/motif_csvs/RBRICS-EXPLLR0.0001-GNNLR0.0001/plots/bbbp_default_motif_bias.png
✅ Saved: ../RBRICS_MINORITY_MP2_DIM16_CORRECTED/plots/motif_csvs/RBRICS-EXPLLR0.0001-GNNLR0.0001/plots/bbbp_GCN_motif_bias.png
✅ Saved: ../RBRICS_MINORITY_MP2_DIM16_CORRECTED/plots/motif_csvs/RBRICS-EXPLLR0.0001-GNNLR0.0001/plots/bbbp_GAT_motif_bias.png
✅ Saved: ../RBRICS_MINORITY_MP2_DIM16_CORRECTED/plots/motif_csvs/RBRICS-EXPLLR0.0001-GNNLR0.0001/plots/bbbp_GIN_motif_bias.png
✅ Saved: ../RBRICS_MINORITY_MP2_DIM16_CORRECTED/plots/motif_csvs/RBRICS-EXPLLR0.0001-GNNLR0.0001/plots/bbbp_SAGE_motif_bias.png
✅ Saved: ../RBRICS_MINORITY_MP2_DIM16_CORRECTED/plots/motif_csvs/RBRICS-EXPLLR0.0001-GNNLR0.0001/plots/benzene_default_m

In [ ]:
input()